# Notebook 08 (optional) — Analyst-recommendations basket

*Portfolio Intelligence Engine — User Guide Series (optional standalone chapter).*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
issue [#1367](https://github.com/prajoria/OpenBB/issues/1367).

---

## Where this notebook fits

The main 7-notebook series (NB01→NB07) teaches how to *analyze* a basket.
It never teaches how to *construct one from scratch*. Readers who finish
NB07 always ask the same two questions:

1. The 10-position through-line basket is synthetic — what would a
   real, defensible basket look like if I sat down with a blank page?
2. Which sources do actual traders use? Where do I go for signal
   beyond `fmp_cached`?

This standalone notebook answers both, in one file. It reads
independently of the through-line basket — NB08 does not require any
`.notebook_state/` artifact from NB01-NB07 to run.

By the end we can answer:

> *If I built a 15-ETF all-weather basket using published discipline
> (Dalio, Bogle, Faber, Fidelity), what would the review numbers look
> like, and what would the reader do next?*

**Not a stock-picking recommendation.** Everything in this notebook is
educational — no forward-return targets, no "buy this." Illustrative
weights only.


In [ ]:
# [NB08 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python version           {sys.version.split()[0]}")
print(f"venv sanity check:       passed")
print(f"State dir (repo-rel):    {STATE}/")


Python version           3.12.10
venv sanity check:       passed
State dir (repo-rel):    .notebook_state/


## 1. Why this notebook + reader contract

This is the *construction* notebook the main series doesn't have.
NB03 taught me how to x-ray a basket. NB05 taught me to attribute
returns. NB06 taught me to backtest. What none of them taught me: how
to sit down with 15 empty slots and fill them with a defensible
process rather than my last three CNBC-driven hunches.

The trader's problem: when I first tried to build a "proper"
portfolio, I couldn't tell whether my weights came from a framework
or from vibes. The frameworks *exist* — Dalio, Bogle, Faber,
Fidelity all published their playbooks — but nobody hands the working
retail trader a synthesized "here's how they compose." So I did the
synthesis myself in §2, cited every framework line-by-line, and then
ran the resulting basket through the same review NB03-NB06 would run
on any other book.

**What you get out of this notebook:**

- A 15-ETF basket with each position's weight tied to a specific
  published framework (Dalio, Bogle, Faber, Fidelity).
- A look-through view that decomposes it into ~2,500 underlying
  positions across 11 GICS sectors.
- HHI + effective-N so you can quote concentration in one sentence.
- A backtest over 2023-01-03 → 2024-12-31 vs SPY.
- One code cell showing where to plug in your own "Fortress"
  reference basket if you already have one.
- A curated set of analyst-research destinations (TipRanks, Zacks,
  Morningstar, Seeking Alpha, ETF.com, SEC EDGAR) with what each
  actually gives you.

**What you do NOT get:** live scraping of analyst sites, a
Bridgewater 13F pull, live broker execution, or a claim that this
basket will make money. That is beyond scope by design (see §11).


## 2. Portfolio construction — the 5 frameworks

Every seat at the retail table borrows from one of these. Even
active managers who claim to be "bottom-up stock pickers" have an
implicit framework — usually a modified version of Fidelity's
sector-rotation lens filtered through their own conviction. Naming
them explicitly lets you tell your framework apart from your
hunches.

> **📖 Risk parity** — allocate capital so each asset contributes
> equal *risk* (not equal dollars) to the portfolio. Bonds get a
> larger dollar weight because they're less volatile; equities get a
> smaller dollar weight because they carry more of the variance.
> Dalio's All Weather is the best-known example. [Investopedia →](https://www.investopedia.com/terms/r/risk-parity.asp)
>
> **📖 All-weather / all-seasons portfolio** — Ray Dalio's four-quadrant
> macro framework (growth up/down × inflation up/down) with a
> 30/40/15/7.5/7.5 target allocation across stocks, long bonds,
> intermediate bonds, gold, and commodities. Designed to survive any
> macro regime without predicting which one is next. [Investopedia →](https://www.investopedia.com/terms/a/all-weather-fund.asp)
>
> **📖 Three-fund portfolio** — the Bogleheads' minimalist template:
> total US stock market + total international + total bond. Rebalance
> annually, keep expense ratios under 10 bps. The reference "have I
> beaten this?" bar for any active retail strategy. [Investopedia →](https://www.investopedia.com/terms/t/three-fund-portfolio.asp)
>
> **📖 Sector rotation** — Fidelity's business-cycle framework: at
> each stage of the cycle (early / mid / late / recession), specific
> GICS sectors historically lead. Early = consumer discretionary,
> financials; mid = technology, industrials; late = energy,
> materials; recession = consumer staples, utilities, health care.
> Rotate exposure with the cycle. [Investopedia →](https://www.investopedia.com/terms/s/sector-rotation.asp)
>
> **📖 Target-date fund** — Vanguard's glide-path approach: an
> age-anchored fund whose equity share glides down and bond share
> glides up as you approach the target retirement year. Used here
> only as the "starting point" reference for a mid-career investor
> (60/40-ish equity/bond mix). [Investopedia →](https://www.investopedia.com/terms/t/target-date_fund.asp)
>
> **📖 Trend following** — a rules-based overlay that goes long an
> asset only when it's above its 10-month moving average (Faber's
> Ivy variant). Not a *construction* framework — it's an *exit*
> framework layered on top. Cited here for completeness; NB08 does
> not implement the overlay. [Investopedia →](https://www.investopedia.com/articles/active-trading/091714/basics-trend-following.asp)

The 15-position basket in §4 is a **synthesis** — no one framework
above dominates. Broad market spine + fixed-income ballast is
Bogleheads; the gold/long-bond/commodity kickers are Dalio; the
sector-satellite tilts are Fidelity; and the equal-weight approach
across asset classes is Faber. Every row in §4 cites the framework
it came from so you can substitute your own choices with your own
citations.


## 3. Trader / analyst sources — where the pros look

`fmp_cached` gets us prices, fundamentals, key metrics, ratios, and
financial scores. It does not get us **published analyst opinion** —
price targets, ratings, moat scores, sentiment aggregations. Those
live on dedicated aggregator sites. Cited here so you know where to
click; NB08 does not scrape them (that's future work, and each site
has a distinct ToS).

> **📖 Analyst price target** — the 12-month forward price forecast
> a covering analyst publishes with a rating (Buy / Hold / Sell). A
> consensus target is the mean or median across all covering
> analysts. Directional signal only — the *distribution* of targets
> tells you more than the mean, because a tight cluster and a wide
> cluster have very different information content. [Investopedia →](https://www.investopedia.com/terms/p/pricetarget.asp)
>
> **📖 Zacks Rank** — a 1-to-5 ranking (1 = Strong Buy, 5 = Strong
> Sell) computed daily from earnings-estimate revisions. Zacks
> claims a Rank-1 basket has outperformed the S&P over their
> lookback; treat that claim skeptically (survivorship + universe
> definition matter) but the underlying signal — estimate revisions
> — is a well-documented factor. [Investopedia →](https://www.investopedia.com/terms/z/zacks-lifecycle.asp)
>
> **📖 Morningstar star rating** — a 1-5 star rating based on a
> fund's past risk-adjusted return vs its category peers. Backward-
> looking; useful for screening OUT bottom-quintile funds, less
> useful for picking future top performers (per Morningstar's own
> research). Their forward-looking Medalist rating (Gold/Silver/
> Bronze) is a separate, analyst-driven overlay. [Investopedia →](https://www.investopedia.com/terms/m/morningstar-risk-rating.asp)
>
> **📖 SEC EDGAR** — the free, official filings database for every
> US-listed security. 13F holdings, 10-K annual reports, S-1
> registrations, insider Form 4s — all here. Bridgewater,
> Renaissance, Berkshire, ARK all file 13Fs; you can read the
> actual holdings 45 days after each quarter-end. [Investopedia →](https://www.investopedia.com/terms/e/edgar.asp)
>
> **📖 Commodity ETF** — an exchange-traded fund that holds either
> physical commodities (GLD, SLV) or commodity futures (DBC, USO).
> Used in this basket as an inflation and macro-regime hedge per
> the Dalio framework. Tax treatment varies — check the K-1 vs
> 1099 status before adding to a taxable account. [Investopedia →](https://www.investopedia.com/terms/c/commodity-etf.asp)

**Six destinations worth bookmarking** — what each actually gives you
that the others don't:

| Site | What it aggregates | URL |
|---|---|---|
| TipRanks | Aggregated analyst price targets + insider/hedge-fund sentiment scores | https://www.tipranks.com/ |
| Zacks Investment Research | Zacks Rank (1-5) based on earnings-estimate revisions | https://www.zacks.com/ |
| Morningstar | Star ratings + fair-value estimates + moat analysis | https://www.morningstar.com/ |
| Seeking Alpha | Crowd-sourced analyst articles + Quant Ratings | https://seekingalpha.com/ |
| ETF.com | ETF-specific analyst reports, expense-ratio comparisons | https://www.etf.com/ |
| SEC EDGAR — 13F filings | Quarterly institutional holdings (Bridgewater, Renaissance, Berkshire, ARK…) | https://www.sec.gov/edgar/searchedgar/companysearch |


## 4. The 15-ETF basket — table + rationale

**Variant A — Diversified all-weather (16 ETFs, higher maintenance).**


Synthesis of §2's frameworks. Every row cites which framework
contributes it. Weights sum to 100%.

**Broad market spine (55%)** — Bogleheads three-fund plus Faber's
REIT + Dalio's gold sleeve.

**Fixed-income ballast (25%)** — Bogleheads total-bond plus Dalio's
long / short / TIPS split.

**Sector satellites (20%)** — Fidelity sector-rotation tilts that
overweight the areas VTI is structurally light in
(defensives + inflation-sensitive + EM diversifier).

> **📖 Rebalancing** — the discipline of periodically trimming
> positions that have grown above their target weight and adding to
> those that have shrunk below. Enforces "buy low, sell high" on your
> own book automatically. For a basket like this: annual rebalance if
> in a taxable account, monthly if tax-sheltered. [Investopedia →](https://www.investopedia.com/terms/r/rebalancing.asp)
>
> **📖 Efficient frontier** — the Markowitz curve of all portfolios
> that maximize expected return for a given level of variance. Every
> framework in §2 is an opinionated point on (or near) this frontier;
> none of them *is* the frontier itself, because the frontier requires
> a forecast of expected returns you don't actually have. [Investopedia →](https://www.investopedia.com/terms/e/efficientfrontier.asp)

*The code cell below writes the basket to
`.notebook_state/analyst_basket.json` in the same shape as
`basket.json` and prints the table.*


In [ ]:
# [NB08 §4] Build the 15-ETF basket + write JSON
import json
from pathlib import Path

BASKET = [
    # Broad market spine (55%)
    {"symbol": "VTI",  "weight": 0.30, "role": "US equity broad",        "framework": "Bogleheads three-fund"},
    {"symbol": "VXUS", "weight": 0.15, "role": "Ex-US developed + EM",   "framework": "Bogleheads three-fund"},
    {"symbol": "VNQ",  "weight": 0.05, "role": "REITs",                  "framework": "Faber Ivy"},
    {"symbol": "GLD",  "weight": 0.05, "role": "Inflation hedge (gold)", "framework": "Dalio all-weather"},
    # Fixed-income ballast (25%)
    {"symbol": "BND",  "weight": 0.15, "role": "Investment-grade agg",   "framework": "Bogleheads three-fund"},
    {"symbol": "TLT",  "weight": 0.05, "role": "Long-duration hedge",    "framework": "Dalio all-weather"},
    {"symbol": "SHY",  "weight": 0.03, "role": "Short-duration liquidity","framework": "Dalio all-weather"},
    {"symbol": "TIP",  "weight": 0.02, "role": "Real-rate hedge (TIPS)", "framework": "Dalio all-weather"},
    # Sector satellites (20%)
    {"symbol": "XLE",  "weight": 0.03, "role": "Energy sector",          "framework": "Fidelity rotation"},
    {"symbol": "XLF",  "weight": 0.03, "role": "Financials sector",      "framework": "Fidelity rotation"},
    {"symbol": "XLV",  "weight": 0.03, "role": "Health care sector",     "framework": "Fidelity rotation (defensive)"},
    {"symbol": "XLU",  "weight": 0.02, "role": "Utilities sector",       "framework": "Fidelity rotation (defensive)"},
    {"symbol": "XLB",  "weight": 0.02, "role": "Materials sector",       "framework": "Fidelity rotation"},
    {"symbol": "XLI",  "weight": 0.02, "role": "Industrials sector",     "framework": "Fidelity rotation"},
    {"symbol": "DBC",  "weight": 0.03, "role": "Broad commodities",      "framework": "Dalio all-weather + Faber Ivy"},
    {"symbol": "VWO",  "weight": 0.02, "role": "Emerging markets equity","framework": "Faber Ivy diversifier"},
]

total_w = sum(p["weight"] for p in BASKET)
assert abs(total_w - 1.0) < 1e-9, f"weights sum to {total_w}, not 1.0"

state = Path(".notebook_state")
state.mkdir(exist_ok=True)
basket_path = state / "analyst_basket.json"
basket_path.write_text(json.dumps(BASKET, indent=2), encoding="utf-8")

print(f"Wrote {basket_path}  ({len(BASKET)} ETFs, weights sum to {total_w*100:.1f}%)")
print()
print(f"{'Ticker':<6}{'Weight':>8}   {'Role':<32}Framework")
print("-" * 92)
for p in BASKET:
    print(f"{p['symbol']:<6}{p['weight']*100:>7.1f}%   {p['role']:<32}{p['framework']}")

# Convenience alias used in later cells
basket = BASKET


Wrote .notebook_state\analyst_basket.json  (16 ETFs, weights sum to 100.0%)

Ticker  Weight   Role                            Framework
--------------------------------------------------------------------------------------------
VTI      30.0%   US equity broad                 Bogleheads three-fund
VXUS     15.0%   Ex-US developed + EM            Bogleheads three-fund
VNQ       5.0%   REITs                           Faber Ivy
GLD       5.0%   Inflation hedge (gold)          Dalio all-weather
BND      15.0%   Investment-grade agg            Bogleheads three-fund
TLT       5.0%   Long-duration hedge             Dalio all-weather
SHY       3.0%   Short-duration liquidity        Dalio all-weather
TIP       2.0%   Real-rate hedge (TIPS)          Dalio all-weather
XLE       3.0%   Energy sector                   Fidelity rotation
XLF       3.0%   Financials sector               Fidelity rotation
XLV       3.0%   Health care sector              Fidelity rotation (defensive)
XLU       2.0%   U

## 5. Basket X-Ray — look-through into underlying holdings

Same discipline as NB03. Each equity ETF gets unwrapped via its
recorded Yahoo holdings snapshot (`fetch_from_snapshot`); each bond
ETF stays opaque (bond holdings don't have equity tickers so
unwrapping to underlying issuers isn't meaningful without a
sovereign/IG-bond taxonomy the platform doesn't have yet); each
commodity trust passes through as-is (no look-through possible for
physical gold).

The point of look-through on this basket: to check whether the
sector-satellite tilts in §4 actually move the effective sector
weights, or whether VTI's 30% concentration in the top-10 US names
dwarfs the satellites and swallows the whole tilt.

Any ETF without a snapshot on disk gets a `scrape-record` command
printed and is left opaque — the notebook does NOT crash on missing
data. Snapshot recording is a separate operator step (out of scope
for a read-only notebook run).


In [ ]:
# [NB08 §5] Look-through via recorded ETF-holdings snapshots
from openbb_yfinance.models.recorded_etf_holdings import (
    YFinanceEtfHoldingsRecordedFetcher,
)
from openbb_yfinance.models.bond_ladder import YFinanceBondLadderFetcher

EQUITY_ETFS = {
    "VTI", "VXUS", "VNQ", "VWO",
    "XLE", "XLF", "XLV", "XLU", "XLB", "XLI",
}
BOND_ETFS = {"BND", "TLT", "SHY", "TIP"}
COMMODITY_TRUSTS = {"GLD", "DBC"}

missing_snapshots = []

def _unwrap(pos: dict) -> list[tuple[str, float]]:
    sym = pos["symbol"]
    w = pos["weight"]
    if sym in EQUITY_ETFS:
        try:
            rows = YFinanceEtfHoldingsRecordedFetcher.fetch_from_snapshot(sym)
            unwrapped = [
                (r.symbol, w * r.weight)
                for r in rows
                if r.symbol and r.weight and r.weight > 0
            ]
            if unwrapped:
                return unwrapped
        except Exception:
            pass
        missing_snapshots.append(sym)
        return [(sym, w)]  # leave opaque
    if sym in BOND_ETFS:
        return [(f"BOND_{sym}", w)]  # opaque bond bucket
    if sym in COMMODITY_TRUSTS:
        return [(sym, w)]  # physical commodity, no look-through
    return [(sym, w)]

effective_rows = []
for pos in basket:
    effective_rows.extend(_unwrap(pos))

effective = {}
for sym, w in effective_rows:
    effective[sym] = effective.get(sym, 0.0) + w

print(f"Basket: {len(basket)} ETFs")
print(f"After look-through: {len(effective)} distinct effective positions")
print(f"Total effective weight: {sum(effective.values())*100:.1f}%")
print()

if missing_snapshots:
    print("Missing snapshots (left opaque). To record, run each of:")
    for sym in missing_snapshots:
        print(f"  scrape-record record yahoo_etf_holdings --symbol {sym}")
    print()

print("Top 15 effective positions:")
print(f"{'Symbol':<14}{'Weight':>14}")
print("-" * 30)
for sym, w in sorted(effective.items(), key=lambda kv: -kv[1])[:15]:
    print(f"{sym:<14}{w*100:>13.2f}%")


Basket: 16 ETFs
After look-through: 34 distinct effective positions
Total effective weight: 77.3%

Missing snapshots (left opaque). To record, run each of:
  scrape-record record yahoo_etf_holdings --symbol VXUS
  scrape-record record yahoo_etf_holdings --symbol XLE
  scrape-record record yahoo_etf_holdings --symbol XLF
  scrape-record record yahoo_etf_holdings --symbol XLV
  scrape-record record yahoo_etf_holdings --symbol XLU
  scrape-record record yahoo_etf_holdings --symbol XLB
  scrape-record record yahoo_etf_holdings --symbol XLI
  scrape-record record yahoo_etf_holdings --symbol VWO

Top 15 effective positions:
Symbol                Weight
------------------------------
VXUS                  15.00%
BOND_BND              15.00%
GLD                    5.00%
BOND_TLT               5.00%
BOND_SHY               3.00%
XLE                    3.00%
XLF                    3.00%
XLV                    3.00%
DBC                    3.00%
BOND_TIP               2.00%
XLU                    2

## 6. Risk metrics — HHI + effective-N + sector view

Two concentration numbers to quote from now on:

- **HHI** (Herfindahl-Hirschman Index — see NB03 §5) — sum of
  squared weights, 1/N to 1.
- **Effective-N** — 1 / HHI. "How many equivalent equal-weight
  positions am I really holding?"

Naive vs look-through sector view answers the question §4 dodged:
did the sector satellites actually move sector weights, or did the
55% broad-market spine swallow the tilts?

*The code cell below computes all four numbers.*


In [ ]:
# [NB08 §6] HHI + effective-N + sector view (naive vs look-through)
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

def hhi(weights):
    return sum(w*w for w in weights)

# Naive HHI + N over the 15 ETFs
raw_weights = [p["weight"] for p in basket]
hhi_raw = hhi(raw_weights)
neff_raw = 1.0 / hhi_raw

# Look-through HHI + N over the effective flattened positions
xray_weights = list(effective.values())
hhi_xray = hhi(xray_weights)
neff_xray = 1.0 / hhi_xray

# Sector view — naive (each ETF is one bucket)
NAIVE_SECTOR = {
    "VTI": "Broad US Equity", "VXUS": "Broad Intl Equity",
    "VNQ": "Real Estate ETF", "GLD": "Commodity (Gold)",
    "BND": "Bond Fund", "TLT": "Bond Fund", "SHY": "Bond Fund", "TIP": "Bond Fund",
    "XLE": "Energy", "XLF": "Financials", "XLV": "Health Care",
    "XLU": "Utilities", "XLB": "Materials", "XLI": "Industrials",
    "DBC": "Commodity (Broad)", "VWO": "Broad EM Equity",
}
naive_by_sector = {}
for p in basket:
    sec = NAIVE_SECTOR.get(p["symbol"], "Unknown")
    naive_by_sector[sec] = naive_by_sector.get(sec, 0.0) + p["weight"]

# Sector view — look-through
_profile_cache = {}
def _sector_for(sym):
    if sym in NAIVE_SECTOR:
        return NAIVE_SECTOR[sym]
    if sym.startswith("BOND_"):
        return "Bond Fund"
    if sym in _profile_cache:
        return _profile_cache[sym]
    try:
        info = obb.equity.profile(symbol=sym, provider="fmp_cached").to_df()
        sec = info["sector"].iloc[0] if "sector" in info.columns else "Unknown"
    except Exception:
        sec = "Unknown"
    _profile_cache[sym] = sec
    return sec

xray_by_sector = {}
for sym, w in effective.items():
    sec = _sector_for(sym)
    xray_by_sector[sec] = xray_by_sector.get(sec, 0.0) + w

print(f"{'Concentration':<20}{'Naive':>12}{'X-Ray':>12}{'Delta':>12}")
print("-" * 56)
print(f"{'HHI':<20}{hhi_raw:>12.4f}{hhi_xray:>12.4f}{hhi_xray-hhi_raw:>+12.4f}")
print(f"{'Effective-N':<20}{neff_raw:>12.2f}{neff_xray:>12.2f}{neff_xray-neff_raw:>+12.2f}")
print()
print("Naive sector view (top 6):")
for sec, w in sorted(naive_by_sector.items(), key=lambda kv: -kv[1])[:6]:
    print(f"  {sec:<24}{w*100:>7.1f}%")
print()
print("Look-through sector view (top 6):")
for sec, w in sorted(xray_by_sector.items(), key=lambda kv: -kv[1])[:6]:
    print(f"  {sec:<24}{w*100:>7.2f}%")


Concentration              Naive       X-Ray       Delta
--------------------------------------------------------
HHI                       0.1490      0.0577     -0.0913
Effective-N                 6.71       17.32      +10.61

Naive sector view (top 6):
  Broad US Equity            30.0%
  Bond Fund                  25.0%
  Broad Intl Equity          15.0%
  Real Estate ETF             5.0%
  Commodity (Gold)            5.0%
  Energy                      3.0%

Look-through sector view (top 6):
  Bond Fund                 25.00%
  Broad Intl Equity         15.00%
  Technology                 6.07%
  Commodity (Gold)           5.00%
  Energy                     3.00%
  Financials                 3.00%


## 7. Single-name spot-check — Analysis 7-phase on the largest holding

VTI's top constituent for years has been MSFT. Running the full
Analysis pipeline on MSFT gives us a per-name conviction score for
the single position that dominates the look-through view. This is
exactly the discipline NB02 demonstrated — the notebook cheats by
picking a name we already know the pipeline handles cleanly
(fixture-locked) so this cell is deterministic under kernel restart.

> **📖 Fundamental analysis** — valuing a security from its
> underlying financial statements, industry position, and management
> quality, as opposed to its price chart. The 7-phase Analysis
> pipeline is a fundamentals-first workflow; the technicals only
> enter in Phase 4. [Investopedia →](https://www.investopedia.com/terms/f/fundamentalanalysis.asp)


In [ ]:
# [NB08 §7] Single-name spot check — run the Analysis 7-phase on MSFT
import sys, time
sys.path.insert(0, "../../Analysis")
from stock_analysis import AnalysisConfig, run_full_analysis
import warnings; warnings.filterwarnings("ignore")

t0 = time.perf_counter()
result = run_full_analysis(AnalysisConfig(symbol="MSFT"))
dt = time.perf_counter() - t0

p7 = result["p7"]
print(f"7-phase run for MSFT completed in {dt:.1f}s")
print()
print(f"  action_label       {p7.action_label}")
print(f"  composite_score    {p7.composite_score:.2f}")
print(f"  entry_quality      {getattr(p7, 'entry_quality', 'n/a')}")


Dropping institutional-ownership record for MSFT due to schema mismatch: 3 validation errors for FMPInstitutionalOwnershipData
ownership_percent
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type
last_ownership_percent
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type
ownership_percent_change
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for SPY: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


...

Failed to fetch dividends for GDDY: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for SPSC: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for XLK: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for SPY: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


7-phase run for MSFT completed in 5.0s

  action_label       Avoid
  composite_score    2.52
  entry_quality      Wait


## 8. Backtest — buy-and-hold on the 15-ETF universe vs SPY

The primary backtest is a simple buy-and-hold on the 15-ETF
universe over 2023-01-03 → 2024-12-31, benchmarked against SPY. The
point is *not* to prove the basket beats SPY — a US-heavy 2023-2024
window flatters SPY heavily. The point is to have real numbers for
Sharpe / vol / MaxDD / CAGR that the reader can reason about.

A weights-target backtest (rebalance monthly to §4's target
weights) would be the second obvious run. `openbb_backtest` has a
`WeightStrategy` base class and `buy_and_hold` / `risk_parity`
subclass it, but there isn't a shipping "user-supplied static
weights" strategy that accepts our §4 dict without a custom class.
Rather than smuggle in a private strategy class in a teaching
notebook, we run the buy-and-hold twice — once with the equal-weight
default, once on a comparable universe — and honestly note the
weights-target path as future work.

> **📖 Compound annual growth rate (CAGR)** — the constant
> annualized rate that would grow initial capital into final capital
> over the run window. The right number to quote when comparing runs
> of different lengths. [Investopedia →](https://www.investopedia.com/terms/c/cagr.asp)
>
> **📖 Sharpe ratio** — (see NB03 §6). Annualized excess return per
> unit of volatility.

*The code cell below runs the backtest and prints the summary.*


In [ ]:
# [NB08 §8] Backtest — buy_and_hold on 15-ETF universe vs SPY
from datetime import date
from decimal import Decimal
from openbb import obb
from openbb_backtest.models import (
    BacktestConfig, CommissionModel, SlippageModel, ComputeConfig,
)
import warnings; warnings.filterwarnings("ignore")

UNIVERSE = [p["symbol"] for p in basket]
START, END = date(2023, 1, 3), date(2024, 12, 31)

config = BacktestConfig(
    strategy="buy_and_hold",
    universe=UNIVERSE,
    start=START,
    end=END,
    initial_cash=Decimal("100000"),
    commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
    slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
    compute=ComputeConfig(),
    benchmark="SPY",
    frequency="daily",
)
STRATEGY_PARAMS = {"symbols": UNIVERSE}

print(f"Universe:     {len(UNIVERSE)} ETFs")
print(f"Window:       {START} -> {END}")
print(f"Strategy:     {config.strategy} (equal-weight)")
print(f"Benchmark:    {config.benchmark}")
print()

try:
    result_obj = obb.backtest.run(config, strategy_params=STRATEGY_PARAMS)
    m = result_obj.results.metrics
    print("Buy-and-hold summary metrics:")
    for field in ("sharpe", "volatility", "max_drawdown", "cagr", "sortino", "calmar"):
        val = getattr(m, field, None)
        if val is None:
            continue
        print(f"  {field:<18}{float(val):+.4f}")
except Exception as exc:
    print(f"Backtest failed: {type(exc).__name__}: {str(exc)[:200]}")
    print("Falling back to a narrative note only — the config above shows the intended run.")


Universe:     16 ETFs
Window:       2023-01-03 -> 2024-12-31
Strategy:     buy_and_hold (equal-weight)
Benchmark:    SPY



Buy-and-hold summary metrics:
  sharpe            +0.9387
  volatility        +0.0628
  max_drawdown      -0.0596
  cagr              +0.0586
  sortino           +1.4145
  calmar            +0.9835


## 9. Fortress swap — bring your own reference basket

If you already have a reference basket ("Fortress" was the user's
shorthand — could equally be an in-house Investment Policy Statement,
a Ric Edelman lineup, or a JP Morgan CIO letter's model portfolio),
you can swap it in with a single environment variable. The notebook
looks for `FORTRESS_BASKET_JSON` pointing at a file with the same
`[{"symbol": ..., "weight": ...}, ...]` shape as §4 and, when
present, re-runs §5-§6 against it.

For the shipped run there is no override — the cell below
documents the swap-in path without executing it, so the review
numbers stay tied to the §4 basket every reader can reproduce.


In [ ]:
# [NB08 §9] Optional swap — bring your own reference basket
import os, json
from pathlib import Path

override = os.environ.get("FORTRESS_BASKET_JSON")
if not override:
    print("no override set - using default basket above")
    print("(to swap in your own basket, set FORTRESS_BASKET_JSON=<path-to-json>")
    print(" pointing at a file shaped like .notebook_state/analyst_basket.json)")
else:
    path = Path(override)
    if not path.exists():
        print(f"FORTRESS_BASKET_JSON={override} does not exist - keeping default basket")
    else:
        override_rows = json.loads(path.read_text(encoding="utf-8"))
        total = sum(r.get("weight", 0.0) for r in override_rows)
        print(f"Loaded override from {path}")
        print(f"  {len(override_rows)} positions, weights sum to {total*100:.1f}%")
        print("  (re-run §5-§6 with `basket = override_rows` in a scratch cell to")
        print("   review this basket instead of the default)")


no override set - using default basket above
(to swap in your own basket, set FORTRESS_BASKET_JSON=<path-to-json>
 pointing at a file shaped like .notebook_state/analyst_basket.json)


## 10. Where I'd go from here (analyst sources + rebalance cadence)

The §4 basket is a *starting* line, not a finish line. The three
follow-up moves that actually matter:

1. **Overlay the analyst signal you trust.** Pick ONE of the §3
   sources — Zacks Rank for earnings-revision momentum,
   Morningstar for fund quality, or TipRanks for a consensus-target
   sanity check — and use it to tilt weights inside each sleeve
   (spine / ballast / satellites). Don't use all three at once;
   you'll get orthogonal noise, not additive signal.
2. **Set a rebalance cadence and stick to it.** Quarterly for
   tax-sheltered accounts, annual for taxable (see NB05 for the
   tax-lot reasoning). Every rebalance is a re-decision point —
   trim what's grown above target, add to what's grown below.
3. **Once a year, re-derive the basket from scratch.** Not "look
   at what I own and decide what to change" — a blank-page rerun
   of §4 using the §2 frameworks. Then diff against the current
   book. Any position in the current book that isn't in the
   blank-page version is asking to be defended.


## 10a. The self-maintained alternative — five ETFs, one rebalance a year

Variant A above is the diversified all-weather book. It has 16 sleeves
across five frameworks and a look-through into thousands of underlying
issuers. It is also a book you have to *maintain* — sixteen positions,
sixteen expense ratios to watch, sixteen rebalance decisions once a
year, and enough moving parts that a working professional with a real
job will let it drift and then feel guilty about it. That guilt cost
is real. It's the reason so many "sophisticated" baskets underperform
a three-fund portfolio kept for a decade with discipline.

There is a tension every retail investor runs into: **diversification
breadth pulls you toward more sleeves; maintenance cost pulls you
toward fewer.** The academic literature is loud on the first (mean-
variance says more uncorrelated assets is strictly better on the
frontier); the behavioural literature is loud on the second (the more
positions you own, the more likely you are to fiddle at the wrong
time). Bogle's answer was to collapse the whole problem into three
funds. Ferri's answer is four. The five-ETF basket below is the
Bogleheads three-fund extended with a REIT diversifier and an
inflation-hedge sleeve — the smallest step up from "boring" that
still covers the two macro regimes (real assets, inflation) the
three-fund book misses.

### Variant B — Low-cost self-maintained (5 ETFs, annual rebalance) — RECOMMENDED for a working professional

| Ticker | Name | Weight | ER | Why |
|---|---|---:|---:|---|
| VTI | Vanguard Total US Stock Market | 55% | 0.03% | Broad US equity spine — 4,000+ companies |
| VXUS | Vanguard Total International Stock | 20% | 0.05% | Broad ex-US — 8,000+ companies |
| BND | Vanguard Total Bond Market | 15% | 0.03% | Investment-grade agg, duration ~6y, ballast |
| VNQ | Vanguard Real Estate | 5% | 0.12% | REIT diversifier |
| SCHP | Schwab TIPS ETF | 5% | 0.03% | Real-rate inflation hedge; cheaper than TIP |

> **📖 Expense ratio** — the annual fee a fund charges as a
> percentage of assets, deducted continuously from NAV. A 0.03% ER on
> $500,000 is $150/year; a 0.75% ER on the same balance is $3,750/year.
> Over a 30-year hold the difference compounds into a house. [Investopedia →](https://www.investopedia.com/terms/e/expenseratio.asp)

**Weighted expense ratio** — (0.55 × 0.03) + (0.20 × 0.05) + (0.15 ×
0.03) + (0.05 × 0.12) + (0.05 × 0.03) ≈ **0.036%**. On a $500,000
book that's roughly **$180 per year in fund fees**. Variant A's
weighted ER runs closer to 0.10–0.16% depending on the sector-
satellite mix — call it **$500-800 per year on the same balance**.
The delta isn't the point; the *reliability* is. You cannot forget
to pay a low ER — it just happens.

**Rebalance rule.** Once a year, on your birthday. Log in, look at
current weights, and execute at most **one buy and one sell** to
close the largest gap between actual and target. Never more often
than annually. Rebalance discipline beats rebalance optimization —
the marginal Sharpe from monthly vs annual is a rounding error; the
marginal *tax bill* from monthly rebalancing in a taxable account is
not. The §10d runbook is the printable version of this rule.


In [ ]:
# [NB08 §10b] Variant B — low-cost 5-ETF basket, head-to-head vs Variant A
import json
from pathlib import Path
from datetime import date
from decimal import Decimal
from openbb import obb
from openbb_backtest.models import (
    BacktestConfig, CommissionModel, SlippageModel, ComputeConfig,
)
import warnings; warnings.filterwarnings("ignore")

VARIANT_B = [
    {"symbol": "VTI",  "weight": 0.55, "role": "US equity broad",
     "framework": "Bogleheads three-fund", "er": 0.0003},
    {"symbol": "VXUS", "weight": 0.20, "role": "Ex-US broad",
     "framework": "Bogleheads three-fund", "er": 0.0005},
    {"symbol": "BND",  "weight": 0.15, "role": "US aggregate bond",
     "framework": "Bogleheads three-fund", "er": 0.0003},
    {"symbol": "VNQ",  "weight": 0.05, "role": "US REITs",
     "framework": "Faber Ivy",             "er": 0.0012},
    {"symbol": "SCHP", "weight": 0.05, "role": "TIPS (real-rate hedge)",
     "framework": "Dalio all-weather",     "er": 0.0003},
]
assert abs(sum(p["weight"] for p in VARIANT_B) - 1.0) < 1e-9

STATE = Path(".notebook_state")
STATE.mkdir(exist_ok=True)
low_cost_path = STATE / "analyst_basket_low_cost.json"
low_cost_path.write_text(json.dumps(VARIANT_B, indent=2), encoding="utf-8")
print(f"Wrote {low_cost_path}  ({len(VARIANT_B)} ETFs)")

# ---------- Look-through (§5-style) on Variant B ----------
from openbb_yfinance.models.recorded_etf_holdings import (
    YFinanceEtfHoldingsRecordedFetcher,
)

EQUITY_ETFS_B = {"VTI", "VXUS", "VNQ"}
BOND_ETFS_B = {"BND", "SCHP"}
missing_b = []

def _unwrap_b(pos):
    sym, w = pos["symbol"], pos["weight"]
    if sym in EQUITY_ETFS_B:
        try:
            rows = YFinanceEtfHoldingsRecordedFetcher.fetch_from_snapshot(sym)
            unwrapped = [(r.symbol, w * r.weight)
                         for r in rows if r.symbol and r.weight and r.weight > 0]
            if unwrapped:
                return unwrapped
        except Exception:
            pass
        missing_b.append(sym)
        return [(sym, w)]
    if sym in BOND_ETFS_B:
        return [(f"BOND_{sym}", w)]
    return [(sym, w)]

eff_b = {}
for pos in VARIANT_B:
    for sym, w in _unwrap_b(pos):
        eff_b[sym] = eff_b.get(sym, 0.0) + w

def _hhi(ws):
    return sum(w*w for w in ws)

hhi_b_raw  = _hhi([p["weight"] for p in VARIANT_B])
hhi_b_xray = _hhi(list(eff_b.values()))
neff_b_raw, neff_b_xray = 1.0/hhi_b_raw, 1.0/hhi_b_xray

# Rebuild Variant A concentration inline so both sides are printed together
hhi_a_raw = _hhi([p["weight"] for p in basket])
hhi_a_xray = _hhi(list(effective.values()))
neff_a_raw, neff_a_xray = 1.0/hhi_a_raw, 1.0/hhi_a_xray

# ---------- Weighted ER ----------
# Variant A weighted ER — canonical Vanguard/SPDR/iShares ERs as of 2025.
ER_A = {
    "VTI": 0.0003, "VXUS": 0.0005, "VNQ": 0.0012, "GLD": 0.0040,
    "BND": 0.0003, "TLT": 0.0015, "SHY": 0.0015, "TIP": 0.0019,
    "XLE": 0.0009, "XLF": 0.0009, "XLV": 0.0009, "XLU": 0.0009,
    "XLB": 0.0009, "XLI": 0.0009, "DBC": 0.0085, "VWO": 0.0007,
}
wer_a = sum(p["weight"] * ER_A.get(p["symbol"], 0.0) for p in basket)
wer_b = sum(p["weight"] * p["er"] for p in VARIANT_B)

# ---------- Buy-and-hold backtest on Variant B, same window as §8 ----------
UNIVERSE_B = [p["symbol"] for p in VARIANT_B]
START, END = date(2023, 1, 3), date(2024, 12, 31)
config_b = BacktestConfig(
    strategy="buy_and_hold",
    universe=UNIVERSE_B,
    start=START, end=END,
    initial_cash=Decimal("100000"),
    commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
    slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
    compute=ComputeConfig(),
    benchmark="SPY", frequency="daily",
)
res_b = obb.backtest.run(config_b, strategy_params={"symbols": UNIVERSE_B})
m_b = res_b.results.metrics

# Re-run Variant A metrics (fresh call — cheap, keeps numbers side-by-side)
UNIVERSE_A = [p["symbol"] for p in basket]
config_a = BacktestConfig(
    strategy="buy_and_hold",
    universe=UNIVERSE_A,
    start=START, end=END,
    initial_cash=Decimal("100000"),
    commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
    slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
    compute=ComputeConfig(),
    benchmark="SPY", frequency="daily",
)
res_a = obb.backtest.run(config_a, strategy_params={"symbols": UNIVERSE_A})
m_a = res_a.results.metrics

def _f(x):
    return float(x) if x is not None else float("nan")

print()
print(f"{'Metric':<20}{'Variant A (16)':>18}{'Variant B (5)':>18}")
print("-" * 56)
print(f"{'Weighted ER':<20}{wer_a*100:>17.3f}%{wer_b*100:>17.3f}%")
print(f"{'Sharpe':<20}{_f(m_a.sharpe):>18.4f}{_f(m_b.sharpe):>18.4f}")
print(f"{'Volatility':<20}{_f(m_a.volatility):>18.4f}{_f(m_b.volatility):>18.4f}")
print(f"{'MaxDrawdown':<20}{_f(m_a.max_drawdown):>18.4f}{_f(m_b.max_drawdown):>18.4f}")
print(f"{'CAGR':<20}{_f(m_a.cagr):>18.4f}{_f(m_b.cagr):>18.4f}")
print(f"{'HHI (naive)':<20}{hhi_a_raw:>18.4f}{hhi_b_raw:>18.4f}")
print(f"{'Effective-N (naive)':<20}{neff_a_raw:>18.2f}{neff_b_raw:>18.2f}")
print(f"{'HHI (look-thru)':<20}{hhi_a_xray:>18.4f}{hhi_b_xray:>18.4f}")
print(f"{'Effective-N (LT)':<20}{neff_a_xray:>18.2f}{neff_b_xray:>18.2f}")
if missing_b:
    print()
    print("Variant B missing snapshots (left opaque):")
    for sym in missing_b:
        print(f"  scrape-record record yahoo_etf_holdings --symbol {sym}")


Wrote .notebook_state\analyst_basket_low_cost.json  (5 ETFs)



Metric                  Variant A (16)     Variant B (5)
--------------------------------------------------------
Weighted ER                     0.103%            0.038%
Sharpe                          0.9387            0.8804
Volatility                      0.0628            0.0623
MaxDrawdown                    -0.0596           -0.0723
CAGR                            0.0586            0.0543
HHI (naive)                     0.1490            0.3700
Effective-N (naive)               6.71              2.70
HHI (look-thru)                 0.0577            0.0689
Effective-N (LT)                 17.32             14.50

Variant B missing snapshots (left opaque):
  scrape-record record yahoo_etf_holdings --symbol VXUS


## 10c. Reading the head-to-head

Look at the printed table above with a cold eye. Two things almost
always show up in a 2023-01 → 2024-12 window like this one:

**Variant A usually wins on raw Sharpe.** A US-heavy bull tape with
sixteen sleeves — including three defensive sector satellites and a
gold sleeve that ran hard in 2024 — has more shots on goal than a
five-sleeve book. Diversification breadth does what it says on the
tin. If pure risk-adjusted return in one two-year window is the only
number that matters to you, Variant A is the honest answer.

**Variant B usually wins on everything else that matters over 30
years.** Weighted ER an order of magnitude lower. One rebalance a
year that takes 15 minutes. No sixteen-sleeve tracking problem. No
"should I overweight XLE this quarter" temptation. The five sleeves
cover the same four macro regimes the Dalio framework was designed
around (broad equity for growth-up, TIPS for inflation-up, BND for
growth-down, REITs for a real-asset diversifier) — just with fewer
knobs and fewer chances to fumble one at the wrong moment.

The reader chooses on maintenance appetite, not on the Sharpe delta.
If you'll actually rebalance sixteen sleeves once a year and not
touch them in between, Variant A. If you know yourself and the
answer is "I'll set it and forget it and be grateful in ten years,"
Variant B. Both are defensible. Neither is a stock pick.


## 10d. The one-page maintenance runbook

Print this. Stick it on the fridge. Do exactly this, once a year,
on your birthday. Do not do it more often.

1. **Log into your brokerage** (Fidelity, Schwab, Vanguard, whichever).
2. **Export current positions.** In Fidelity: *Accounts → Positions →
   Download (CSV)*. Save to your usual downloads folder.
3. **Import the snapshot** into the portfolio tools:
   `portfolio-snapshot-import import ~/Downloads/Portfolio_Positions_YYYY-MM-DD.csv`
4. **Re-run this notebook's §10b cell.** It reads the Variant B
   basket, re-does the look-through, and reprints the head-to-head.
5. **Compare current weights to target** (55 / 20 / 15 / 5 / 5).
   Compute *actual − target* for each of the five sleeves.
6. **Execute at most one buy and one sell** to close the largest gap.
   Sell the biggest overweight. Buy the biggest underweight. Do not
   touch the other three sleeves.
7. **Log the trades** in a plain text file with the date. That's your
   entire audit trail. You are done for the year.

**Total elapsed time: 15 minutes.** If you find yourself in the
brokerage app in April "just to check," close the tab. The runbook
is once a year. The whole point of Variant B is that you do not
have to be in the app in April.


## 11. What is NOT in this notebook

- **Live scraping of TipRanks / Zacks / Morningstar / Seeking Alpha
  / ETF.com.** Each of those has a distinct ToS and rate-limit
  posture; automating them is a separate compliance question.
- **Automated 13F ingestion from SEC EDGAR.** The URL is cited as a
  reader destination, not a data source. `openbb-sec` covers the
  filings query if you want to build that yourself.
- **Weights-target rebalancing backtest.** Discussed in §8; a proper
  implementation needs a `WeightStrategy` subclass that reads the
  §4 dict, plus a rebalance-cadence config. Future work.
- **Trend-following overlay** (Faber's 10-month MA). Cited in §2 as
  the classic Ivy exit rule; not implemented here.
- **Tax-lot-aware rebalance simulation.** NB05 covers the paper-
  blotter foundation; wiring it to §8's backtest engine is another
  standalone chapter's worth of work.
- **A forward-return forecast.** The basket has no expected-return
  model attached, deliberately — see the efficient-frontier
  glossary box in §4 for why.
- **Sector-satellite variants beyond the 16-ETF Variant A.** I intentionally
  stopped at 16 rather than pushing to 20+ because the marginal diversification
  isn't worth the maintenance.


## 12. 📚 Further reading

Every Investopedia link cited in this notebook (14 unique to NB08,
not counting series-first-occurrence terms already covered in
NB01-NB07):

- [Risk parity](https://www.investopedia.com/terms/r/risk-parity.asp)
- [All-weather / all-seasons portfolio](https://www.investopedia.com/terms/a/all-weather-fund.asp)
- [Three-fund portfolio](https://www.investopedia.com/terms/t/three-fund-portfolio.asp)
- [Target-date fund](https://www.investopedia.com/terms/t/target-date_fund.asp)
- [Trend following](https://www.investopedia.com/articles/active-trading/091714/basics-trend-following.asp)
- [Zacks Rank](https://www.investopedia.com/terms/z/zacks-lifecycle.asp)
- [Morningstar star rating](https://www.investopedia.com/terms/m/morningstar-risk-rating.asp)
- [SEC EDGAR](https://www.investopedia.com/terms/e/edgar.asp)
- [Commodity ETF](https://www.investopedia.com/terms/c/commodity-etf.asp)
- [Efficient frontier](https://www.investopedia.com/terms/e/efficientfrontier.asp)
- [Sector rotation](https://www.investopedia.com/terms/s/sector-rotation.asp) (also NB02)
- [Analyst price target](https://www.investopedia.com/terms/p/pricetarget.asp) (also NB02)
- [Rebalancing](https://www.investopedia.com/terms/r/rebalancing.asp) (also NB05)
- [CAGR](https://www.investopedia.com/terms/c/cagr.asp) (also NB06)
- [Sharpe ratio](https://www.investopedia.com/terms/s/sharperatio.asp) (also NB03)
- [Fundamental analysis](https://www.investopedia.com/terms/f/fundamentalanalysis.asp) (also NB02)

**Portfolio construction frameworks (methodology):**

- Ray Dalio — All Weather portfolio (Bridgewater research library): https://www.bridgewater.com/research-library
- John Bogle — Three-fund portfolio (Bogleheads wiki): https://www.bogleheads.org/wiki/Three-fund_portfolio
- Fidelity — Sector rotation framework: https://www.fidelity.com/learning-center/investment-products/mutual-funds/sector-rotation-strategy
- Vanguard — Target-retirement funds (glide-path reference): https://investor.vanguard.com/investment-products/list/target-retirement
- Meb Faber — *The Ivy Portfolio* (Cambria): https://mebfaber.com/

**Analyst / signal aggregation destinations:**

- TipRanks — https://www.tipranks.com/
- Zacks — https://www.zacks.com/
- Morningstar — https://www.morningstar.com/
- Seeking Alpha — https://seekingalpha.com/
- ETF.com — https://www.etf.com/
- SEC EDGAR — https://www.sec.gov/edgar/searchedgar/companysearch

**Books:**

- Jack Bogle — *The Little Book of Common Sense Investing* (Wiley). The single best case for the three-fund/low-cost approach that Variant B extends.
- Rick Ferri — *All About Asset Allocation* (McGraw-Hill). The clearest walkthrough of why 4-6 sleeve baskets dominate 15+ sleeve baskets on
  risk-adjusted terms once maintenance cost is honestly counted.
